In [4]:
import whisper
import librosa
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [5]:
ROOT = Path("../")

AUDIO_DIR = ROOT / "data" / "processed"

REFERENCE_DIR = ROOT / "data" / "references"

REFERENCE_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
print("Cargando Whisper Medium...")

model = whisper.load_model("medium")

print("Modelo cargado correctamente.")

Cargando Whisper Medium...


100%|█████████████████████████████████████| 1.42G/1.42G [02:32<00:00, 9.99MiB/s]


Modelo cargado correctamente.


In [7]:
audio_files = sorted(AUDIO_DIR.glob("*.wav"))

print(f"Total de audios: {len(audio_files)}")

Total de audios: 50


In [8]:
#Generar transcripciones para cada archivo de audio
import time

transcripciones = []

for audio_path in tqdm(audio_files):

    try:

        # Leer el audio con librosa
        audio, sr = librosa.load(
            audio_path,
            sr=16000,
            mono=True
        )

        inicio = time.perf_counter()

        resultado = model.transcribe(
            audio,
            language="es",
            fp16=False
        )

        fin = time.perf_counter()

        transcripciones.append({

            "ID_Audio": audio_path.name,

            "Tiempo_Whisper_s": round(fin-inicio,3),

            "Texto_Whisper_Medium": resultado["text"].strip(),

            "Texto_Corregido_Manual": resultado["text"].strip()

        })

    except Exception as e:

        print(audio_path.name, e)

  0%|          | 0/50 [00:00<?, ?it/s]d:\User\Escritorio\tareas\Ciclo 8\ProyectoFinal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 50/50 [41:45<00:00, 50.11s/it] 


In [9]:
df = pd.DataFrame(transcripciones)

df.head()

,ID_Audio,Tiempo_Whisper_s,Texto_Whisper_Medium,Texto_Corregido_Manual
0,spontaneous-speech-es-71834.wav,51.044,"Mi festividad favorita es la navidad, se feste...","Mi festividad favorita es la navidad, se feste..."
1,spontaneous-speech-es-71835.wav,49.770,¿Qué mascotas son las preferidas en donde vive...,¿Qué mascotas son las preferidas en donde vive...
2,spontaneous-speech-es-74952.wav,38.364,"espinaca, selga, zanahoria, carnes rojas","espinaca, selga, zanahoria, carnes rojas"
3,spontaneous-speech-es-74953.wav,38.205,en algún momento entre la primaria y la secund...,en algún momento entre la primaria y la secund...
4,spontaneous-speech-es-74954.wav,37.387,y por mi región normalmente los niños juegan a...,y por mi región normalmente los niños juegan a...


In [10]:
#Guardad CSV
csv_path = REFERENCE_DIR / "transcripciones_referencia.csv"

df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Archivo guardado en:\n{csv_path}")

Archivo guardado en:
..\data\references\transcripciones_referencia.csv


In [11]:
#Visualizar algunas filas del DataFrame
print(df.sample(5))

                           ID_Audio  Tiempo_Whisper_s  \
19  spontaneous-speech-es-83306.wav            41.051   
15  spontaneous-speech-es-80762.wav            38.106   
9   spontaneous-speech-es-74959.wav            40.386   
25  spontaneous-speech-es-83330.wav            36.187   
45  spontaneous-speech-es-83381.wav            31.967   

                                 Texto_Whisper_Medium  \
19  demostrando educación y respeto contra todas l...   
15  Elementos nutritivos pueden ser frutas y verdu...   
9   generalmente una botella de vino, una caja de ...   
25                           cosa mayor, Oreo y Vera.   
45                        jugar o jugar con la tablet   

                               Texto_Corregido_Manual  
19  demostrando educación y respeto contra todas l...  
15  Elementos nutritivos pueden ser frutas y verdu...  
9   generalmente una botella de vino, una caja de ...  
25                           cosa mayor, Oreo y Vera.  
45                        jugar o 